# Text-to-SQL Assistant (Northwind)

This project converts plain-English business questions into SQL, runs the SQL safely on a database, and explains the result in plain English. It is built with **Python, SQLite, the Gemini API and Streamlit**, and it is tested on 20 questions with known correct answers.

## Key features

- **Automatic schema reading:** tables, columns, primary keys, foreign keys and date ranges are read from the database and given to the model.
- **Two layers of safety:** a text check allows only a single `SELECT` statement, and the database is opened read-only.
- **Self-correction:** if the generated SQL fails, the model sees the error and gets one retry. The fixed SQL goes through the safety check again.
- **Evaluation:** 20 questions (easy, medium, hard) are scored by comparing results with reference SQL.
- **Web app:** a Streamlit app lets anyone ask questions, with an option to upload their own SQLite or CSV file.

## How it works

```
Question -> Prompt with schema -> Gemini writes SQL -> Safety check -> Run on read-only database
                                                                          |
                                         on error: one retry (checked again) -> Plain-English answer
```

## Requirements

- Google Colab, with two secrets added in the key icon of the left sidebar (notebook access on): `GEMINI_API_KEY` and `NGROK_TOKEN`
- A free Gemini API key from Google AI Studio and a free ngrok account (ngrok is only needed for Step 10)

## Notebook outline

| Step | Content |
|---|---|
| 0 | Setup: install the SDK and download the database |
| 1 | Explore the database and confirm it is read-only |
| 2 | Schema extraction |
| 3 | Prompt design |
| 4 | Connect to Gemini and generate SQL |
| 5 | Safety checks |
| 6 | Execute queries and self-correct errors |
| 7 | Plain-English answers |
| 8 | Complete pipeline |
| 9 | Evaluation on 20 test questions |
| 10 | Streamlit web app |
| 11 | Summary and limitations |

# Step 0: Setup

## 0.1 Install dependencies

Installs the Google Gemini SDK, which the notebook uses to generate SQL and plain-English answers.

In [ ]:
!pip install google-genai -q

## 0.2 Download the database

Northwind is a standard sample database for a trading company. It holds customers, orders, products, employees, suppliers and shippers. The SQLite version is downloaded from GitHub.

In [ ]:
import urllib.request

# Download the SQLite version of the Northwind database from GitHub
url = "https://raw.githubusercontent.com/jpwhite3/northwind-SQLite3/main/dist/northwind.db"
urllib.request.urlretrieve(url, "northwind.db")
print("Database downloaded successfully!")

Database downloaded successfully!


**Result:** The database file was downloaded and saved as `northwind.db`.

# Step 1: Explore the database

## 1.1 Connect to the database (read-only)

The connection uses `mode=ro` (read-only). Even if a harmful query got past the text check, the database itself would refuse to change.

In [ ]:
import sqlite3
import pandas as pd

# Open the database in read-only mode (mode=ro)
conn = sqlite3.connect("file:northwind.db?mode=ro", uri=True)
cursor = conn.cursor()
print("Connected (read-only)")

Connected (read-only)


**Result:** The connection is open and set to read-only.

## 1.2 Verify that the database is protected

This attempts a harmless write (a price set to its own value). The expected result is **WRITE BLOCKED**.

In [ ]:
# A protected database must reject this write attempt
try:
    conn.execute("UPDATE Products SET UnitPrice = UnitPrice WHERE ProductID = 1")
    print("WRITE WORKED -> database is NOT protected")
except Exception as e:
    print("WRITE BLOCKED ->", e)

WRITE BLOCKED -> attempt to write a readonly database


**Result:** The write attempt was rejected with *attempt to write a readonly database*. This confirms the second layer of protection works.

## 1.3 List all tables

In [ ]:
# List every table stored in the database
cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table';")
tables = cursor.fetchall()

for t in tables:
    print(t[0])

Categories
sqlite_sequence
CustomerCustomerDemo
CustomerDemographics
Customers
Employees
EmployeeTerritories
Order Details
Orders
Products
Regions
Shippers
Suppliers
Territories


**Result:** The database contains 13 tables plus SQLite's internal `sqlite_sequence`. `Order Details` has a space in its name, so it needs double quotes in SQL. This is also why the prompt in Step 3 includes a rule about quoting such names.

## 1.4 Preview sample rows

In [ ]:
# Preview the first rows of the three core tables
print("--- Customers ---")
print(pd.read_sql_query("SELECT CustomerId, CompanyName, Country FROM Customers LIMIT 3;", conn))

print("\n--- Orders ---")
print(pd.read_sql_query("SELECT OrderId, CustomerId, EmployeeID, OrderDate FROM Orders LIMIT 3;", conn))

print("\n--- Order Details ---")
print(pd.read_sql_query("SELECT * FROM 'Order Details' LIMIT 3;", conn))

--- Customers ---
  CustomerID                         CompanyName  Country
0      ALFKI                 Alfreds Futterkiste  Germany
1      ANATR  Ana Trujillo Emparedados y helados   Mexico
2      ANTON             Antonio Moreno Taquería   Mexico

--- Orders ---
   OrderID CustomerID  EmployeeID   OrderDate
0    10248      VINET           5  2016-07-04
1    10249      TOMSP           6  2016-07-05
2    10250      HANAR           4  2016-07-08

--- Order Details ---
   OrderID  ProductID  UnitPrice  Quantity  Discount
0    10248         11       14.0        12       0.0
1    10248         42        9.8        10       0.0
2    10248         72       34.8         5       0.0


**Result:** Each customer has a text ID (for example `ALFKI`) that links to its orders. Orders are linked to employees. `Order Details` holds one row per product in an order, with the unit price, quantity and discount. This table is the basis for revenue, calculated as `UnitPrice * Quantity * (1 - Discount)`.

# Step 2: Schema extraction

The model can only write correct SQL if it knows the database structure. The helpers below read the structure directly from the database, so nothing is typed by hand.

## 2.1 Find tables that contain data

In [ ]:
# Return only the tables that contain at least one row
def get_tables_with_data(conn):
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table' AND name != 'sqlite_sequence';")
    all_tables = [row[0] for row in cursor.fetchall()]

    tables_with_data = []
    for table in all_tables:
        cursor.execute(f"SELECT COUNT(*) FROM '{table}';")
        count = cursor.fetchone()[0]
        if count > 0:
            tables_with_data.append(table)

    return tables_with_data

## 2.2 Describe each table's columns

In [ ]:
# Describe every table as text: column names, data types and primary keys
def get_schema_text(conn, tables):
    cursor = conn.cursor()
    schema_lines = []
    for table in tables:
        cursor.execute(f'PRAGMA table_info("{table}");')
        columns = cursor.fetchall()
        col_descriptions = []
        for col in columns:
            col_name, col_type, is_pk = col[1], col[2], col[5]
            desc = f"{col_name} ({col_type})"
            if is_pk:
                desc += " [PRIMARY KEY]"
            col_descriptions.append(desc)
        schema_lines.append(f'TABLE "{table}":')
        schema_lines.append("  " + ", ".join(col_descriptions))
        schema_lines.append("")
    return "\n".join(schema_lines)

## 2.3 Detect table relationships (foreign keys)

In [ ]:
# Read the foreign keys so the model knows how tables are joined
def get_relationships_text(conn, tables):
    cursor = conn.cursor()
    lines = []
    for table in tables:
        cursor.execute(f'PRAGMA foreign_key_list("{table}");')
        for fk in cursor.fetchall():
            table_name = f'"{table}"' if " " in table else table
            lines.append(f'- {table_name}.{fk[3]} -> {fk[2]}.{fk[4]}')
    if not lines:
        return ""
    return "Relationships (foreign keys):\n" + "\n".join(lines)

## 2.4 Detect date columns

In [ ]:
# Find the latest value of each date column so that phrases like "last quarter"
# are measured against the data's own timeline, not today's date
def get_date_context(conn, tables):
    cursor = conn.cursor()
    date_notes = []
    for table in tables:
        cursor.execute(f'PRAGMA table_info("{table}");')
        for col in cursor.fetchall():
            col_name = col[1]
            if "date" in col_name.lower():
                cursor.execute(f'SELECT MAX("{col_name}") FROM "{table}";')
                max_val = cursor.fetchone()[0]
                if max_val:
                    date_notes.append(f"- {table}.{col_name}: most recent value is '{max_val}'")
    if not date_notes:
        return ""
    return "\nDate reference points (use these instead of today's real date):\n" + "\n".join(date_notes)

## 2.5 Verify the four helpers

In [ ]:
# Run all four helpers once to confirm they work
tables = get_tables_with_data(conn)
print("Tables:", tables)
print()
print(get_schema_text(conn, tables)[:300])
print()
print(get_relationships_text(conn, tables))
print()
print(get_date_context(conn, tables))

Tables: ['Categories', 'Customers', 'Employees', 'EmployeeTerritories', 'Order Details', 'Orders', 'Products', 'Regions', 'Shippers', 'Suppliers', 'Territories']

TABLE "Categories":
  CategoryID (INTEGER) [PRIMARY KEY], CategoryName (TEXT), Description (TEXT), Picture (BLOB)

TABLE "Customers":
  CustomerID (TEXT) [PRIMARY KEY], CompanyName (TEXT), ContactName (TEXT), ContactTitle (TEXT), Address (TEXT), City (TEXT), Region (TEXT), PostalCode (TEXT), Country

Relationships (foreign keys):
- Employees.ReportsTo -> Employees.EmployeeID
- EmployeeTerritories.TerritoryID -> Territories.TerritoryID
- EmployeeTerritories.EmployeeID -> Employees.EmployeeID
- "Order Details".ProductID -> Products.ProductID
- "Order Details".OrderID -> Orders.OrderID
- Orders.ShipVia -> Shippers.ShipperID
- Orders.CustomerID -> Customers.CustomerID
- Orders.EmployeeID -> Employees.EmployeeID
- Products.SupplierID -> Suppliers.SupplierID
- Products.CategoryID -> Categories.CategoryID
- Territories.RegionID -> R

**Result:** The helpers found 11 tables with data (the other tables are empty and are skipped), listed every column with its type and primary key, detected 11 foreign-key relationships, and found the most recent value of each date column. The latest `OrderDate` is 2023-10-28, so "last quarter" refers to July to September 2023, not to the current date.

# Step 3: Prompt design

## 3.1 Build the system prompt

The system prompt gives the model the schema, the table relationships, the date reference points and a fixed set of rules (SELECT only, exact names, raw SQL output).

In [ ]:
# Combine schema, relationships, date context and rules into the model's instructions
def build_system_prompt(conn):
    tables = get_tables_with_data(conn)

    # Collect all database context
    full_schema = get_schema_text(conn, tables)
    relationships = get_relationships_text(conn, tables)
    if relationships:
        full_schema += "\n" + relationships

    date_context = get_date_context(conn, tables)

    return f"""You are a SQL assistant for a SQLite database.

Your job: convert a plain-English question into a single, valid SQLite SELECT query.

DATABASE SCHEMA:
{full_schema}
{date_context}

RULES:
1. Only write SELECT queries. Never write INSERT, UPDATE, DELETE, DROP, ALTER, or anything else that changes data.
2. Use exactly the table and column names shown above (case-sensitive).
3. If a table name has a space in it, wrap it in double quotes.
4. Return ONLY the raw SQL. No explanation, no markdown fences, no extra text.
5. If the question can't be answered with this schema, return exactly: NO_QUERY_POSSIBLE
6. If a question needs two separate results combined (like "highest and lowest"), use UNION ALL with each half wrapped as its own subquery, like: SELECT * FROM (SELECT ... ORDER BY ... LIMIT 1) UNION ALL SELECT * FROM (SELECT ... ORDER BY ... LIMIT 1). Never write two full statements separated by a semicolon.
"""

## 3.2 Inspect the prompt

In [ ]:
prompt = build_system_prompt(conn)
print(prompt)

You are a SQL assistant for a SQLite database.

Your job: convert a plain-English question into a single, valid SQLite SELECT query.

DATABASE SCHEMA:
TABLE "Categories":
  CategoryID (INTEGER) [PRIMARY KEY], CategoryName (TEXT), Description (TEXT), Picture (BLOB)

TABLE "Customers":
  CustomerID (TEXT) [PRIMARY KEY], CompanyName (TEXT), ContactName (TEXT), ContactTitle (TEXT), Address (TEXT), City (TEXT), Region (TEXT), PostalCode (TEXT), Country (TEXT), Phone (TEXT), Fax (TEXT)

TABLE "Employees":
  EmployeeID (INTEGER) [PRIMARY KEY], LastName (TEXT), FirstName (TEXT), Title (TEXT), TitleOfCourtesy (TEXT), BirthDate (DATE), HireDate (DATE), Address (TEXT), City (TEXT), Region (TEXT), PostalCode (TEXT), Country (TEXT), HomePhone (TEXT), Extension (TEXT), Photo (BLOB), Notes (TEXT), ReportsTo (INTEGER), PhotoPath (TEXT)

TABLE "EmployeeTerritories":
  EmployeeID (INTEGER) [PRIMARY KEY], TerritoryID (TEXT) [PRIMARY KEY]

TABLE "Order Details":
  OrderID (INTEGER) [PRIMARY KEY], ProductI

**Result:** The prompt contains the schema, the foreign-key relationships, the date reference points and the six rules. This text is sent to the model with every question.

# Step 4: Connect to the AI model

## 4.1 Connect to Gemini

The API key is read from Colab Secrets (key icon in the left sidebar), so it is never written in the code.

In [ ]:
from google.colab import userdata
from google import genai

# Read the API key from Colab Secrets and create the Gemini client
api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

MODEL = "gemini-3.5-flash-lite"

## 4.2 Retry helper

The free Gemini plan allows about 15 requests per minute. When the limit is exceeded, Google returns an error 429 with the time to wait. This helper reads that time, waits, and tries again.

In [ ]:
import re
import time

# Call Gemini and retry on failure; on a rate-limit error (429), wait as long as Google asks
def call_gemini_with_retry(model, contents, config=None, max_retries=5):
    for attempt in range(max_retries):
        try:
            if config:
                return client.models.generate_content(model=model, contents=contents, config=config)
            return client.models.generate_content(model=model, contents=contents)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            message = str(e)
            if "429" in message or "RESOURCE_EXHAUSTED" in message:
                # Read the suggested wait time from the error message (capped at 90 seconds)
                found = re.search(r"retry in ([\d.]+)s", message)
                wait_seconds = (float(found.group(1)) + 3) if found else 30
                wait_seconds = min(wait_seconds, 90)
                print(f"Free-plan limit reached. Waiting {wait_seconds:.0f}s, then trying again...")
            else:
                wait_seconds = 5 * (attempt + 1)
                print(f"AI call failed ({e}). Retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)

## 4.3 Convert a question into SQL

In [ ]:
# Send a plain-English question to the model and return the SQL it writes
def question_to_sql(question, system_prompt):
    response = call_gemini_with_retry(
        model=MODEL,
        contents=question,
        config={"system_instruction": system_prompt}
    )
    sql = response.text.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()  # in case the AI adds fences anyway
    return sql

## 4.4 Test question-to-SQL

In [ ]:
# Example: generate SQL for one question
system_prompt = build_system_prompt(conn)
sql = question_to_sql("Which employee sold the most last quarter?", system_prompt)
print(sql)

SELECT e.FirstName, e.LastName
FROM Employees e
JOIN Orders o ON e.EmployeeID = o.EmployeeID
JOIN "Order Details" od ON o.OrderID = od.OrderID
WHERE o.OrderDate >= '2023-07-01' AND o.OrderDate < '2023-10-01'
GROUP BY e.EmployeeID
ORDER BY SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) DESC
LIMIT 1;


**Result:** For the question *Which employee sold the most last quarter?* the model joined `Employees`, `Orders` and `Order Details`, limited the dates to July to September 2023, ranked employees by discounted revenue and kept the top one.

# Step 5: Safety checks

AI-generated SQL is never executed without validation. The project uses two layers of protection.

## 5.1 Layer 1: text validation

The check accepts a single SELECT statement and rejects anything else. Layer 2 is the read-only database connection from step 1.1.

In [ ]:
import re

# Layer 1 of the safety checks: allow a single SELECT statement and nothing else
FORBIDDEN_WORDS = ["DROP", "DELETE", "UPDATE", "INSERT", "ALTER", "TRUNCATE", "REPLACE", "ATTACH", "DETACH", "PRAGMA", "CREATE"]


def is_safe_select(sql: str) -> bool:
    if not sql or not sql.strip():
        return False

    cleaned = sql.strip()
    # The model returns this marker when the question cannot be answered
    if cleaned == "NO_QUERY_POSSIBLE":
        return False

    stripped = cleaned.rstrip(";").strip()
    if ";" in stripped:          # blocks stacking a second statement
        return False

    # The statement must start with SELECT
    if not re.match(r"^\s*SELECT\b", stripped, re.IGNORECASE):
        return False

    # Reject any statement that contains a data-changing keyword
    tokens = re.findall(r"[A-Za-z]+", stripped.upper())
    if any(word in FORBIDDEN_WORDS for word in tokens):
        return False

    return True

## 5.2 Test the validator with tricky cases

In [ ]:
# Expected results: True, False, False, False, True, False
test_cases = [
    "SELECT * FROM Customers;",
    "SELECT * FROM Orders WHERE OrderID = 1; DROP TABLE Orders;",  # sneaky
    "DROP TABLE Customers;",
    "UPDATE Products SET UnitPrice = 0;",
    "SELECT * FROM Products WHERE ProductName = 'test'",
    "NO_QUERY_POSSIBLE",
]

for sql in test_cases:
    print(f"{is_safe_select(sql)!s:<6} | {sql}")

True   | SELECT * FROM Customers;
False  | SELECT * FROM Orders WHERE OrderID = 1; DROP TABLE Orders;
False  | DROP TABLE Customers;
False  | UPDATE Products SET UnitPrice = 0;
True   | SELECT * FROM Products WHERE ProductName = 'test'
False  | NO_QUERY_POSSIBLE


**Result:** The two ordinary SELECT statements are accepted (`True`). The stacked statement ending in `DROP TABLE`, the plain `DROP`, the `UPDATE` and the no-answer marker are all rejected (`False`).

# Step 6: Execute queries and self-correct errors

## 6.1 Run a query safely

In [ ]:
# Run SQL and return (result, error) so that a failed query never crashes the program
def run_query(sql, conn):
    try:
        df = pd.read_sql_query(sql, conn)
        return df, None
    except Exception as e:
        return None, str(e)

## 6.2 Test query execution

In [ ]:
# A valid query should return a table and no error
df, error = run_query("SELECT CompanyName, Country FROM Customers LIMIT 3;", conn)
print(df)
print("Error:", error)

                          CompanyName  Country
0                 Alfreds Futterkiste  Germany
1  Ana Trujillo Emparedados y helados   Mexico
2             Antonio Moreno Taquería   Mexico
Error: None


**Result:** A valid query returns a table and `Error: None`.

## 6.3 Demonstrate a failing query

The column name below contains a typo. Without a correction step, a user would see this error whenever the model makes a similar mistake.

In [ ]:
# A deliberately wrong column name, to see what an error looks like
bad_sql = "SELECT CompanyNme FROM Customers LIMIT 3"   # typo: CompanyNme
df, error = run_query(bad_sql, conn)
print("Error:", error)

Error: Execution failed on sql 'SELECT CompanyNme FROM Customers LIMIT 3': no such column: CompanyNme


**Result:** SQLite reports `no such column: CompanyNme`. The error is returned as text instead of stopping the program, so it can be passed to the model for correction.

## 6.4 Self-correction

The model is shown the question, the SQL that failed and the error message, and is asked to correct its own SQL.

In [ ]:
# Give the model its question, failed SQL and the error message so it can correct itself
def fix_sql(question, bad_sql, error_message, system_prompt):
    prompt = f"""The user asked: "{question}"

You wrote this SQL:
{bad_sql}

SQLite gave this error:
{error_message}

Write a corrected SQL query. Follow the same rules as before. Return ONLY the raw SQL."""

    response = call_gemini_with_retry(
        model=MODEL,
        contents=prompt,
        config={"system_instruction": system_prompt}
    )
    sql = response.text.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql

## 6.5 Test the correction on the typo from 6.3

In [ ]:
# Ask the model to repair the typo from step 6.3, then run the corrected SQL
system_prompt = build_system_prompt(conn)
fixed = fix_sql("Show 3 customer company names", bad_sql, error, system_prompt)
print(fixed)
print()
print(run_query(fixed, conn)[0])

SELECT CompanyName FROM Customers LIMIT 3

                          CompanyName
0                 Alfreds Futterkiste
1  Ana Trujillo Emparedados y helados
2             Antonio Moreno Taquería


**Result:** The model corrected `CompanyNme` to `CompanyName`, and the corrected query ran successfully.

# Step 7: Plain-English answers

## 7.1 Explain the result

The model turns the result table into a short answer. If the result has more than 20 rows, it is told the real total and must state that the list is partial.

In [ ]:
# Only the first 20 rows are sent to the model, which keeps the prompt small
MAX_ROWS_FOR_AI = 20


# Turn a result table into a short plain-English answer
def explain_answer(question, df):
    total_rows = len(df)
    preview_df = df.head(MAX_ROWS_FOR_AI)
    result_text = preview_df.to_string(index=False)

    # If the result is longer than 20 rows, tell the model to say the list is partial
    cut_off_note = ""
    if total_rows > MAX_ROWS_FOR_AI:
        cut_off_note = (
            f"\nIMPORTANT: the full result has {total_rows} rows, but only the first "
            f"{MAX_ROWS_FOR_AI} are shown above. Say clearly that the list is partial "
            f"(first {MAX_ROWS_FOR_AI} of {total_rows}).\n"
        )

    prompt = f"""A user asked: "{question}"

Here is the data that answers it:
{result_text}
{cut_off_note}
Write the answer following these formatting rules:
- If the answer involves MULTIPLE items (a list of customers, products, etc.), format it as a bulleted markdown list, one item per line.
- If the answer is a SINGLE value or fact, write it as 1 plain sentence.
- Format dollar amounts with a $ sign, commas for thousands, and 2 decimal places.
- Don't mention SQL, tables, or dataframes."""

    response = call_gemini_with_retry(model=MODEL, contents=prompt)
    return response.text.strip()

## 7.2 Test with a short result

In [ ]:
# Short result: expect a bulleted list
df, error = run_query("SELECT CompanyName FROM Customers LIMIT 3;", conn)
print(explain_answer("What are the first 3 customers?", df))

* Alfreds Futterkiste
* Ana Trujillo Emparedados y helados
* Antonio Moreno Taquería


**Result:** A short result is returned as a clean bulleted list.

## 7.3 Test with a long result (more than 20 rows)

In [ ]:
# Long result: expect a note that only part of the list is shown
df, error = run_query("SELECT CompanyName FROM Customers;", conn)
print("Rows in the result:", len(df))
print()
print(explain_answer("List all customers.", df))

Rows in the result: 93

Here is the list of customers (showing the first 20 of 93):

* Alfreds Futterkiste
* Ana Trujillo Emparedados y helados
* Antonio Moreno Taquería
* Around the Horn
* Berglunds snabbköp
* Blauer See Delikatessen
* Blondesddsl père et fils
* Bólido Comidas preparadas
* Bon app'
* Bottom-Dollar Markets
* B's Beverages
* Cactus Comidas para llevar
* Centro comercial Moctezuma
* Chop-suey Chinese
* Comércio Mineiro
* Consolidated Holdings
* Drachenblut Delikatessen
* Du monde entier
* Eastern Connection
* Ernst Handel


**Result:** The query returned 93 rows. The answer lists only the first 20 and states that it is showing 20 of 93, as intended.

# Step 8: Complete pipeline

## 8.1 Combine the steps into two functions

`get_result` does the core work: SQL generation, safety check, execution and one retry. `answer_question` calls it and then writes the plain-English answer. The evaluation in Step 9 uses `get_result` directly, which needs fewer model calls.

In [ ]:
# Pipeline: question -> SQL -> safety check -> run -> one validated retry on error
def get_result(question, conn):
    """Returns (df, sql, error, retried)."""
    system_prompt = build_system_prompt(conn)
    sql = question_to_sql(question, system_prompt)

    if not is_safe_select(sql):
        return None, sql, f"Blocked or invalid query: {sql}", False

    df, run_error = run_query(sql, conn)
    retried = False
    if run_error:
        retried = True
        fixed_sql = fix_sql(question, sql, run_error, system_prompt)

        # never trust new AI output without checking it again
        if not is_safe_select(fixed_sql):
            return None, fixed_sql, f"The fixed query was blocked: {fixed_sql}", True

        sql = fixed_sql
        df, run_error = run_query(sql, conn)
        if run_error:
            return None, sql, f"SQL failed even after one retry: {run_error}", True

    return df, sql, None, retried


# Full flow: get the result, then explain it in plain English
def answer_question(question, conn):
    df, sql, error, retried = get_result(question, conn)
    if error:
        return None, None, sql, error

    if df.empty:
        return df, "No data matched that question.", sql, None

    answer = explain_answer(question, df)
    return df, answer, sql, None

## 8.2 Test with valid and invalid questions

In [ ]:
# Two valid questions, one off-topic question and one attempt to modify data
good_and_bad_questions = [
    "Which employee sold the most last quarter?",   # should work
    "What are the top 3 products by revenue?",      # should work
    "What's the weather like today?",               # nonsense
    "Delete all customers from Germany",            # tries to modify data
]

for question in good_and_bad_questions:
    print(f"\nQ: {question}")
    df, answer, sql, error = answer_question(question, conn)
    if error:
        print("Handled safely:", error)
    else:
        print("Answer:", answer)


Q: Which employee sold the most last quarter?
Answer: Steven Buchanan sold the most last quarter.

Q: What are the top 3 products by revenue?
Answer: Here are the top 3 products by revenue:

* Côte de Blaye
* Thüringer Rostbratwurst
* Mishi Kobe Niku

Q: What's the weather like today?
Handled safely: Blocked or invalid query: NO_QUERY_POSSIBLE

Q: Delete all customers from Germany
Handled safely: Blocked or invalid query: NO_QUERY_POSSIBLE


**Result:** The two valid questions were answered correctly in plain English. For the weather question and the *Delete all customers* request the model returned `NO_QUERY_POSSIBLE`, which the safety check treats as invalid. Both were handled with a message and nothing was run or changed.

# Step 9: Evaluation on 20 test questions

This step measures how reliable the system is. It uses 20 plain-English questions, each with a hand-written **reference SQL**. Every reference query was run against the database first, so each expected answer is known to be valid. The model answers the same 20 questions, and its results are compared with the reference results.

The questions range from easy to hard. Questions with tied answers were avoided on purpose (for example, two suppliers sharing first place), so each question has one clear correct answer.

## 9.1 Questions and reference SQL

In [ ]:
# (id, level, question, correct_sql, does_row_order_matter)
EVAL_SET = [
 (1, "easy", "How many customers are there?",
  "SELECT COUNT(*) FROM Customers", False),
 (2, "easy", "How many orders have been placed in total?",
  "SELECT COUNT(*) FROM Orders", False),
 (3, "easy", "List the names of all product categories.",
  "SELECT CategoryName FROM Categories", False),
 (4, "easy", "How many products are discontinued?",
  "SELECT COUNT(*) FROM Products WHERE Discontinued = '1'", False),
 (5, "easy", "Which products are out of stock?",
  "SELECT ProductName FROM Products WHERE UnitsInStock = 0", False),
 (6, "easy", "What is the average freight cost per order?",
  "SELECT AVG(Freight) FROM Orders", False),
 (7, "medium", "Which country has the most customers?",
  "SELECT Country FROM Customers GROUP BY Country ORDER BY COUNT(*) DESC LIMIT 1", True),
 (8, "medium", "What are the 5 most expensive products?",
  "SELECT ProductName FROM Products ORDER BY UnitPrice DESC LIMIT 5", True),
 (9, "medium", "How many orders were placed in 2022?",
  "SELECT COUNT(*) FROM Orders WHERE OrderDate >= '2022-01-01' AND OrderDate < '2023-01-01'", False),
 (10, "medium", "How many orders did each shipping company deliver? Show the company name and the count.",
  "SELECT s.CompanyName, COUNT(*) FROM Orders o JOIN Shippers s ON o.ShipVia = s.ShipperID GROUP BY s.CompanyName", False),
 (11, "medium", "Which category has the most products?",
  "SELECT c.CategoryName FROM Products p JOIN Categories c ON p.CategoryID = c.CategoryID GROUP BY c.CategoryName ORDER BY COUNT(*) DESC LIMIT 1", True),
 (12, "medium", "What is the average unit price of products in each category? Show the category name and the average price.",
  "SELECT c.CategoryName, AVG(p.UnitPrice) FROM Products p JOIN Categories c ON p.CategoryID = c.CategoryID GROUP BY c.CategoryName", False),
 (13, "medium", "Which employee has handled the most orders? Give the first and last name.",
  "SELECT e.FirstName, e.LastName FROM Orders o JOIN Employees e ON o.EmployeeID = e.EmployeeID GROUP BY e.EmployeeID ORDER BY COUNT(*) DESC LIMIT 1", True),
 (14, "medium", "What is the most expensive product in the Beverages category?",
  "SELECT p.ProductName FROM Products p JOIN Categories c ON p.CategoryID = c.CategoryID WHERE c.CategoryName = 'Beverages' ORDER BY p.UnitPrice DESC LIMIT 1", True),
 (15, "hard", "What is the total revenue from all orders after discounts?",
  "SELECT SUM(UnitPrice * Quantity * (1 - Discount)) FROM \"Order Details\"", False),
 (16, "hard", "Who are the top 3 customers by total revenue? Show the company name.",
  "SELECT c.CompanyName FROM Customers c JOIN Orders o ON c.CustomerID = o.CustomerID JOIN \"Order Details\" d ON o.OrderID = d.OrderID GROUP BY c.CustomerID ORDER BY SUM(d.UnitPrice * d.Quantity * (1 - d.Discount)) DESC LIMIT 3", True),
 (17, "hard", "What was the total revenue in each year? Show the year and the revenue.",
  "SELECT STRFTIME('%Y', o.OrderDate), SUM(d.UnitPrice * d.Quantity * (1 - d.Discount)) FROM Orders o JOIN \"Order Details\" d ON o.OrderID = d.OrderID GROUP BY STRFTIME('%Y', o.OrderDate)", False),
 (18, "hard", "How many customers have never placed an order?",
  "SELECT COUNT(*) FROM Customers WHERE CustomerID NOT IN (SELECT CustomerID FROM Orders)", False),
 (19, "hard", "Which employee sold the most last quarter? Give the first and last name.",
  "SELECT e.FirstName, e.LastName FROM Employees e JOIN Orders o ON e.EmployeeID = o.EmployeeID JOIN \"Order Details\" d ON o.OrderID = d.OrderID WHERE o.OrderDate >= '2023-07-01' AND o.OrderDate < '2023-10-01' GROUP BY e.EmployeeID ORDER BY SUM(d.UnitPrice * d.Quantity * (1 - d.Discount)) DESC LIMIT 1", True),
 (20, "hard", "Which product has generated the most revenue? Show the product name.",
  "SELECT p.ProductName FROM Products p JOIN \"Order Details\" d ON p.ProductID = d.ProductID GROUP BY p.ProductID ORDER BY SUM(d.UnitPrice * d.Quantity * (1 - d.Discount)) DESC LIMIT 1", True),
]

print("Number of test questions:", len(EVAL_SET))

Number of test questions: 20


## 9.2 Validate the reference queries

Before testing the model, confirm that all 20 reference queries run. If one failed, the problem would be in the test set, not in the model.

In [ ]:
# Confirm every reference query runs before it is used as the ground truth
for qid, level, question, ref_sql, order_matters in EVAL_SET:
    ref_df = pd.read_sql_query(ref_sql, conn)
    print(f"Q{qid:>2} OK  ({len(ref_df)} row(s))  {question}")

Q 1 OK  (1 row(s))  How many customers are there?
Q 2 OK  (1 row(s))  How many orders have been placed in total?
Q 3 OK  (8 row(s))  List the names of all product categories.
Q 4 OK  (1 row(s))  How many products are discontinued?
Q 5 OK  (5 row(s))  Which products are out of stock?
Q 6 OK  (1 row(s))  What is the average freight cost per order?
Q 7 OK  (1 row(s))  Which country has the most customers?
Q 8 OK  (5 row(s))  What are the 5 most expensive products?
Q 9 OK  (1 row(s))  How many orders were placed in 2022?
Q10 OK  (3 row(s))  How many orders did each shipping company deliver? Show the company name and the count.
Q11 OK  (1 row(s))  Which category has the most products?
Q12 OK  (8 row(s))  What is the average unit price of products in each category? Show the category name and the average price.
Q13 OK  (1 row(s))  Which employee has handled the most orders? Give the first and last name.
Q14 OK  (1 row(s))  What is the most expensive product in the Beverages category?
Q15 OK  

**Result:** All 20 reference queries run successfully, so the test set is valid.

## 9.3 Scoring method

The SQL text is not compared, because different queries can return the same answer. The **results** are compared:

- The model's result must have the **same number of rows** as the reference result.
- **Every value** in the reference result must appear in the model's result. Extra columns (such as an ID next to a name) are allowed.
- Numbers are compared to 2 decimal places, so `37.98` and `37.979166` are equal. Text is compared without regard to capitals or surrounding spaces.
- Row order is not compared.

In [ ]:
# Normalise values so that numbers and text can be compared fairly
def clean(v):
    try:
        return round(float(v), 2)      # numbers: compare to 2 decimals
    except (TypeError, ValueError):
        return str(v).strip().lower()  # text: ignore capital letters and spaces


# Compare the AI's result with the reference result
def results_match(ref_df, ai_df, order_matters=False):
    # 1) the AI must return the same number of rows
    if ai_df is None or len(ref_df) != len(ai_df):
        return False

    # 2) every value in the correct result must appear in the AI result
    #    (the AI may add extra columns, that is fine)
    ai_values = [clean(v) for v in ai_df.values.flatten()]
    for v in ref_df.values.flatten():
        if clean(v) not in ai_values:
            return False
    return True

## 9.4 Validate the scoring function

A checker that passes everything is useless. It must return True for the correct result and False for the wrong ones.

In [ ]:
# One correct result and three results that are close but wrong
ref = pd.read_sql_query("SELECT ProductName FROM Products ORDER BY UnitPrice DESC LIMIT 5", conn)
same_but_extra_column = pd.read_sql_query("SELECT ProductName, UnitPrice FROM Products ORDER BY UnitPrice DESC LIMIT 5", conn)
wrong_top5 = pd.read_sql_query("SELECT ProductName FROM Products ORDER BY UnitPrice ASC LIMIT 5", conn)
wrong_count = pd.read_sql_query("SELECT ProductName FROM Products ORDER BY UnitPrice DESC LIMIT 3", conn)

print("Right answer            ->", results_match(ref, ref, True), "(want True)")
print("Right + extra column    ->", results_match(ref, same_but_extra_column, True), "(want True)")
print("Wrong products          ->", results_match(ref, wrong_top5, True), "(want False)")
print("Wrong number of rows    ->", results_match(ref, wrong_count, True), "(want False)")

Right answer            -> True (want True)
Right + extra column    -> True (want True)
Wrong products          -> False (want False)
Wrong number of rows    -> False (want False)


**Result:** The scoring function behaves as expected. It accepts the correct result and a result with an extra column, and it rejects a result with the wrong products and a result with the wrong number of rows.

## 9.5 Run the evaluation

This asks the model all 20 questions, with an 8-second pause between questions to stay within the free limit of 15 requests per minute. It takes about 3 to 5 minutes.

If the run is interrupted, run the cell again: finished questions are kept and the run continues from the next one. To start from zero, run `results = []` first.

In [ ]:
# Keep earlier scores if this cell is re-run, so finished questions are skipped
if "results" not in globals():
    results = []

already_done = {r["id"] for r in results}

# Ask the AI every question, score it against the reference SQL and record the outcome
for qid, level, question, ref_sql, order_matters in EVAL_SET:
    if qid in already_done:
        print(f"Q{qid:>2} already done, skipping")
        continue

    ref_df = pd.read_sql_query(ref_sql, conn)
    ai_df, ai_sql, error, retried = get_result(question, conn)

    if error:
        status = "ERROR"
    elif results_match(ref_df, ai_df, order_matters):
        status = "PASS"
    else:
        status = "FAIL"

    results.append({
        "id": qid, "level": level, "question": question, "status": status,
        "retried": retried, "ai_sql": ai_sql, "correct_sql": ref_sql, "error": error,
    })
    note = " (fixed its own SQL)" if retried and status == "PASS" else ""
    print(f"Q{qid:>2} [{level:<6}] {status:<5} {question}{note}")

    # Pause between questions to stay within the free rate limit
    time.sleep(8)

Q 1 [easy  ] PASS  How many customers are there?
Q 2 [easy  ] PASS  How many orders have been placed in total?
Q 3 [easy  ] PASS  List the names of all product categories.
Q 4 [easy  ] PASS  How many products are discontinued?
Q 5 [easy  ] PASS  Which products are out of stock?
Q 6 [easy  ] PASS  What is the average freight cost per order?
Q 7 [medium] PASS  Which country has the most customers?
Q 8 [medium] PASS  What are the 5 most expensive products?
Q 9 [medium] PASS  How many orders were placed in 2022?
Q10 [medium] PASS  How many orders did each shipping company deliver? Show the company name and the count.
Q11 [medium] PASS  Which category has the most products?
Q12 [medium] PASS  What is the average unit price of products in each category? Show the category name and the average price.
Q13 [medium] PASS  Which employee has handled the most orders? Give the first and last name.
Q14 [medium] PASS  What is the most expensive product in the Beverages category?
Q15 [hard  ] PASS  Wha

**Result:** The model passed all 20 questions.

## 9.6 Score

The score covers the 20 Northwind questions above only. The upload feature of the app was not part of this test.

In [ ]:
# Summarise the pass rate overall and by difficulty level
results_df = pd.DataFrame(results)
passed = (results_df["status"] == "PASS").sum()
total = len(results_df)

print(f"SCORE: {passed}/{total}  ({passed / total * 100:.0f}%)")
print()
print(results_df.groupby("level")["status"].apply(lambda s: f"{(s == 'PASS').sum()}/{len(s)}").reindex(["easy", "medium", "hard"]))
print()
print("Questions where the AI fixed its own SQL and then passed:",
      int(((results_df["retried"]) & (results_df["status"] == "PASS")).sum()))

results_df.to_csv("eval_results.csv", index=False)
print("\nSaved full details to eval_results.csv")

SCORE: 20/20  (100%)

level
easy      6/6
medium    8/8
hard      6/6
Name: status, dtype: object

Questions where the AI fixed its own SQL and then passed: 0

Saved full details to eval_results.csv


**Result:** The model scored 20/20 (100%): 6/6 easy, 8/8 medium and 6/6 hard. No question needed the self-correction step. The full details were saved to `eval_results.csv`. Scores can vary slightly between runs because model output is not fully deterministic.

## 9.7 Review failed questions

Failures are the most useful part of an evaluation. For each one, compare the two SQL queries and decide whether the model misread the question or the reference SQL is only one of several valid readings.

In [ ]:
# Show full details only for the questions that did not pass
failures = results_df[results_df["status"] != "PASS"]

if failures.empty:
    print("The AI passed all 20 questions.")
else:
    for _, r in failures.iterrows():
        print("=" * 70)
        print(f"Q{r['id']} [{r['level']}] {r['status']}: {r['question']}")
        print("\nCORRECT SQL:\n" + r["correct_sql"])
        print("\nAI SQL:\n" + str(r["ai_sql"]))
        if r["error"]:
            print("\nERROR:", r["error"])

The AI passed all 20 questions.


**Result:** There are no failed questions in this run.

# Step 10: Streamlit web app

The pipeline is packaged as a web app so that anyone can ask questions without opening the notebook.

## 10.1 Stop any running app

In [ ]:
# Stop any Streamlit server left running from an earlier run
!pkill -f streamlit

## 10.2 Write the app file

This cell writes the complete app to `app.py`. It reuses the same logic as the notebook: read-only database, safety checks, one validated retry and the partial-list note. It also lets a user upload their own SQLite or CSV file. The cell prints `Writing app.py` (or `Overwriting app.py` on a re-run).

In [ ]:
%%writefile app.py
import re
import time
import sqlite3
import tempfile
import urllib.request
import os

import pandas as pd
import streamlit as st
from google import genai


# ---------------- Settings ----------------
DB_PATH = "northwind.db"
MODEL = "gemini-3.5-flash-lite"
SQLITE_MAGIC = b"SQLite format 3\x00"


# ---------------- Database access ----------------
def open_northwind():
    # read-only connection: the database itself refuses to change
    return sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, check_same_thread=False)


def ensure_database_exists():
    if not os.path.exists(DB_PATH):
        url = "https://raw.githubusercontent.com/jpwhite3/northwind-SQLite3/main/dist/northwind.db"
        urllib.request.urlretrieve(url, DB_PATH)


# ---------------- Schema helpers (same as Step 2) ----------------
def get_tables_with_data(conn):
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name != 'sqlite_sequence';")
    all_tables = [row[0] for row in cursor.fetchall()]
    tables_with_data = []
    for table in all_tables:
        cursor.execute(f'SELECT COUNT(*) FROM "{table}";')
        if cursor.fetchone()[0] > 0:
            tables_with_data.append(table)
    return tables_with_data


def get_schema_text(conn, tables):
    cursor = conn.cursor()
    schema_lines = []
    for table in tables:
        cursor.execute(f'PRAGMA table_info("{table}");')
        columns = cursor.fetchall()
        col_descriptions = []
        for col in columns:
            col_name, col_type, is_pk = col[1], col[2], col[5]
            desc = f"{col_name} ({col_type})"
            if is_pk:
                desc += " [PRIMARY KEY]"
            col_descriptions.append(desc)
        schema_lines.append(f'TABLE "{table}":')
        schema_lines.append("  " + ", ".join(col_descriptions))
        schema_lines.append("")
    return "\n".join(schema_lines)


def get_relationships_text(conn, tables):
    cursor = conn.cursor()
    lines = []
    for table in tables:
        cursor.execute(f'PRAGMA foreign_key_list("{table}");')
        for fk in cursor.fetchall():
            table_name = f'"{table}"' if " " in table else table
            lines.append(f'- {table_name}.{fk[3]} -> {fk[2]}.{fk[4]}')
    if not lines:
        return ""
    return "Relationships (foreign keys):\n" + "\n".join(lines)


def get_date_context(conn, tables):
    cursor = conn.cursor()
    date_notes = []
    for table in tables:
        cursor.execute(f'PRAGMA table_info("{table}");')
        for col in cursor.fetchall():
            col_name = col[1]
            if "date" in col_name.lower():
                cursor.execute(f'SELECT MAX("{col_name}") FROM "{table}";')
                max_val = cursor.fetchone()[0]
                if max_val:
                    date_notes.append(f"- {table}.{col_name}: most recent value is '{max_val}'")
    if not date_notes:
        return ""
    return "\nDate reference points (use these instead of today's real date):\n" + "\n".join(date_notes)


# ---------------- Prompt and AI calls (same as Steps 3 and 4) ----------------
def build_system_prompt(conn):
    tables = get_tables_with_data(conn)
    full_schema = get_schema_text(conn, tables)
    relationships = get_relationships_text(conn, tables)
    if relationships:
        full_schema += "\n" + relationships
    date_context = get_date_context(conn, tables)

    return f"""You are a SQL assistant for a SQLite database.

Your job: convert a plain-English question into a single, valid SQLite SELECT query.

DATABASE SCHEMA:
{full_schema}
{date_context}

RULES:
1. Only write SELECT queries. Never write INSERT, UPDATE, DELETE, DROP, ALTER, or anything else that changes data.
2. Use exactly the table and column names shown above (case-sensitive).
3. If a table name has a space in it, wrap it in double quotes.
4. Return ONLY the raw SQL. No explanation, no markdown fences, no extra text.
5. If the question can't be answered with this schema, return exactly: NO_QUERY_POSSIBLE
6. If a question needs two separate results combined (like "highest and lowest"), use UNION ALL with each half wrapped as its own subquery, like: SELECT * FROM (SELECT ... ORDER BY ... LIMIT 1) UNION ALL SELECT * FROM (SELECT ... ORDER BY ... LIMIT 1). Never write two full statements separated by a semicolon.
"""


def call_gemini_with_retry(client, model, contents, config=None, max_retries=5):
    for attempt in range(max_retries):
        try:
            if config:
                return client.models.generate_content(model=model, contents=contents, config=config)
            return client.models.generate_content(model=model, contents=contents)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            message = str(e)
            if "429" in message or "RESOURCE_EXHAUSTED" in message:
                # free plan limit: wait as long as Google asks, then try again
                found = re.search(r"retry in ([\d.]+)s", message)
                wait_seconds = (float(found.group(1)) + 3) if found else 30
                time.sleep(min(wait_seconds, 90))
            else:
                time.sleep(5 * (attempt + 1))


def question_to_sql(question, system_prompt, client):
    response = call_gemini_with_retry(client, MODEL, question, config={"system_instruction": system_prompt})
    sql = response.text.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql


# ---------------- Safety check (same as Step 5) ----------------
FORBIDDEN_WORDS = ["DROP", "DELETE", "UPDATE", "INSERT", "ALTER", "TRUNCATE", "REPLACE", "ATTACH", "DETACH", "PRAGMA", "CREATE"]


def is_safe_select(sql: str) -> bool:
    if not sql or not sql.strip():
        return False
    cleaned = sql.strip()
    if cleaned == "NO_QUERY_POSSIBLE":
        return False
    stripped = cleaned.rstrip(";").strip()
    if ";" in stripped:
        return False
    if not re.match(r"^\s*SELECT\b", stripped, re.IGNORECASE):
        return False
    tokens = re.findall(r"[A-Za-z]+", stripped.upper())
    if any(word in FORBIDDEN_WORDS for word in tokens):
        return False
    return True


# ---------------- Run, fix and explain (same as Steps 6 to 8) ----------------
def run_query(sql, conn):
    try:
        df = pd.read_sql_query(sql, conn)
        return df, None
    except Exception as e:
        return None, str(e)


# give the AI one chance to correct its own SQL
def fix_sql(question, bad_sql, error_message, system_prompt, client):
    prompt = f"""The user asked: "{question}"

You wrote this SQL:
{bad_sql}

SQLite gave this error:
{error_message}

Write a corrected SQL query. Follow the same rules as before. Return ONLY the raw SQL."""
    response = call_gemini_with_retry(client, MODEL, prompt, config={"system_instruction": system_prompt})
    sql = response.text.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql


# tell the AI (and the user) when the result is longer than what is sent
MAX_ROWS_FOR_AI = 20


def explain_answer(question, df, client):
    total_rows = len(df)
    preview_df = df.head(MAX_ROWS_FOR_AI)
    result_text = preview_df.to_string(index=False)

    cut_off_note = ""
    if total_rows > MAX_ROWS_FOR_AI:
        cut_off_note = (
            f"\nIMPORTANT: the full result has {total_rows} rows, but only the first "
            f"{MAX_ROWS_FOR_AI} are shown above. Say clearly that the list is partial "
            f"(first {MAX_ROWS_FOR_AI} of {total_rows}).\n"
        )

    prompt = f"""A user asked: "{question}"

Here is the data that answers it:
{result_text}
{cut_off_note}
Write the answer following these formatting rules:
- If the answer involves MULTIPLE items (a list of customers, products, etc.), format it as a bulleted markdown list, one item per line.
- If the answer is a SINGLE value or fact, write it as 1 plain sentence.
- Format dollar amounts with a $ sign, commas for thousands, and 2 decimal places.
- Don't mention SQL, tables, or dataframes."""
    response = call_gemini_with_retry(client, MODEL, prompt)
    return response.text.strip()


def get_result(question, conn, client):
    """Returns (df, sql, error, retried)."""
    system_prompt = build_system_prompt(conn)
    sql = question_to_sql(question, system_prompt, client)

    if not is_safe_select(sql):
        return None, sql, f"Blocked or invalid query: {sql}", False

    df, run_error = run_query(sql, conn)
    retried = False
    if run_error:
        retried = True
        fixed_sql = fix_sql(question, sql, run_error, system_prompt, client)
        if not is_safe_select(fixed_sql):
            return None, fixed_sql, f"The fixed query was blocked: {fixed_sql}", True
        sql = fixed_sql
        df, run_error = run_query(sql, conn)
        if run_error:
            return None, sql, f"SQL failed even after one retry: {run_error}", True

    return df, sql, None, retried


def answer_question(question, conn, client):
    df, sql, error, retried = get_result(question, conn, client)
    if error:
        return None, None, sql, error
    if df.empty:
        return df, "No data matched that question.", sql, None
    answer = explain_answer(question, df, client)
    return df, answer, sql, None


# ---------------- UPLOAD: load your own SQLite or CSV file (read-only) ----------------
def load_uploaded_db(file_bytes):
    if file_bytes[:16] != SQLITE_MAGIC:
        return None, "That doesn't look like a valid SQLite database file."
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".db")
    tmp.write(file_bytes)
    tmp.close()
    try:
        # read-only
        conn = sqlite3.connect(f"file:{tmp.name}?mode=ro", uri=True, check_same_thread=False)
        tables = get_tables_with_data(conn)
        if not tables:
            return None, "This database file has no tables with data in it."
        return conn, None
    except Exception as e:
        return None, f"Couldn't open this database file: {e}"


def load_uploaded_csv(uploaded_file):
    try:
        first_line = uploaded_file.readline()
        if isinstance(first_line, bytes):
            first_line = first_line.decode("utf-8", errors="strict")
        raw_columns = [c.strip() for c in first_line.strip().split(",")]
        uploaded_file.seek(0)
    except Exception:
        return None, "Couldn't read this file. Please upload a clean CSV with a proper header row."

    if len(raw_columns) != len(set(raw_columns)):
        return None, "This file has duplicate column names. Please upload a clean CSV with unique headers."

    try:
        df = pd.read_csv(uploaded_file)
    except Exception:
        return None, "Couldn't read this file. Please upload a clean CSV with a proper header row."

    if df.empty:
        return None, "This file has no data rows."

    try:
        conn = sqlite3.connect(":memory:", check_same_thread=False)
        df.to_sql("uploaded_data", conn, index=False)
        # after loading, only reading is allowed
        conn.execute("PRAGMA query_only = ON")
        return conn, None
    except Exception as e:
        return None, f"Couldn't load this data: {e}"


# ---------------- STREAMLIT UI ----------------
st.set_page_config(page_title="Text-to-SQL Assistant (Northwind)", page_icon="🗄️")
st.title("🗄️ Text-to-SQL Assistant")
st.caption("Ask a plain-English question about the Northwind business database — or upload your own.")

# Northwind loads automatically every time — no upload needed to try it
if "conn" not in st.session_state:
    ensure_database_exists()
    st.session_state.conn = open_northwind()
    st.session_state.db_label = "Northwind (default)"
    st.session_state.is_northwind = True



with st.sidebar:
    typed_key = st.text_input("Gemini API key (optional)", type="password",
                              help="Leave empty to use the demo key, or paste your own. Free, no credit card - get one at aistudio.google.com/apikey")
    # the demo key stays on the server and is never sent to the visitor's browser
    api_key = typed_key or os.environ.get("GEMINI_API_KEY", "")

    st.markdown("---")
    st.subheader("📊 Active dataset")
    dataset_label_box = st.empty()   # filled in below, after the upload / reset code has run

    st.markdown("---")
    st.subheader("ℹ️ About this project")
    st.markdown("""
This tool turns plain-English questions into real SQL, runs it safely
against a database, and explains the answer in plain English.

**How it works:**
1. Reads the database's real structure automatically
2. Gemini writes the SQL
3. Every query is checked — only SELECT allowed, and the database is opened read-only
4. If the SQL fails, the AI gets one chance to fix it
5. The result gets explained back in plain English

Built with Python, SQLite, Streamlit, and the Gemini API.
""")



with st.expander("📁 Use your own data instead (optional)"):
    st.caption("Upload a SQLite (.db) file or a CSV with a proper header row. Clean files work best. Please don't upload confidential data: your question, the table structure and a sample of the results are sent to Gemini.")

    uploaded_file = st.file_uploader("Upload", type=["db", "sqlite", "sqlite3", "csv"])

    # load a file only once, so "Reset to Northwind" is not undone by the next click
    file_id = (uploaded_file.name, uploaded_file.size) if uploaded_file is not None else None
    if uploaded_file is not None and st.session_state.get("loaded_file") != file_id:
        if uploaded_file.name.endswith(".csv"):
            new_conn, error = load_uploaded_csv(uploaded_file)
        else:
            new_conn, error = load_uploaded_db(uploaded_file.getvalue())

        if error:
            st.error(error)
        else:
            st.session_state.conn = new_conn
            st.session_state.db_label = uploaded_file.name
            st.session_state.is_northwind = False
            st.session_state.loaded_file = file_id
            st.success(f"Using: {uploaded_file.name}")

    if st.button("Reset to Northwind (default)"):
        ensure_database_exists()
        st.session_state.conn = open_northwind()
        st.session_state.db_label = "Northwind (default)"
        st.session_state.is_northwind = True

    st.caption(f"Active dataset: **{st.session_state.db_label}**")

    if st.session_state.is_northwind:
        st.markdown("---")
        st.markdown("**Try asking:**")
        st.markdown("- Which employee sold the most last quarter?")
        st.markdown("- What are the top 3 products by revenue?")
        st.markdown("- Which category has the most products?")

dataset_label_box.info(f"**{st.session_state.db_label}**")

if "question_text" not in st.session_state:
    st.session_state.question_text = ""

def set_question(q):
    st.session_state.question_text = q

if st.session_state.is_northwind:
    st.markdown("**Try one of these:**")
    col1, col2, col3 = st.columns(3)
    with col1:
        st.button("Top employee", on_click=set_question, args=("Which employee sold the most last quarter?",))
    with col2:
        st.button("Top products", on_click=set_question, args=("What are the top 3 products by revenue?",))
    with col3:
        st.button("Top category", on_click=set_question, args=("Which category has the most products?",))

question = st.text_input("Your question:", value=st.session_state.question_text,
                          placeholder="e.g. Which employee sold the most last quarter?")
ask_clicked = st.button("Ask")



if ask_clicked:
    if not api_key:
        st.error("Please enter your Gemini API key in the sidebar.")
    elif not question.strip():
        st.warning("Type a question first.")
    else:
        client = genai.Client(api_key=api_key)
        with st.spinner("Thinking..."):
            df, answer, sql, error = answer_question(question, st.session_state.conn, client)
        if error:
            st.error(error)
        else:
            st.success(answer)
            if df is not None and len(df) > MAX_ROWS_FOR_AI:
                st.caption(f"The explanation above uses the first {MAX_ROWS_FOR_AI} of {len(df)} rows. "
                           f"Open the section below to see all rows.")
            with st.expander("See the generated SQL and raw result"):
                st.code(sql, language="sql")
                if df is not None and not df.empty:
                    st.dataframe(df)

Writing app.py


**Result:** `app.py` is written. It contains the full pipeline and the web interface: a question box, example questions, an optional upload of a SQLite or CSV file, and an expandable section that shows the generated SQL and the full result table.

## 10.3 Install the web app tools

In [ ]:
# Streamlit builds the app; pyngrok gives it a public web address
!pip install streamlit pyngrok -q

## 10.4 Connect ngrok

ngrok provides a public web address for the app running in Colab. The token is read from Colab Secrets, so it is never written in the code. Add a secret named `NGROK_TOKEN` (with notebook access turned on) before running this cell.

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Register the ngrok token stored in Colab Secrets
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))

## 10.5 Start the app

In [ ]:
import os
import subprocess
from google.colab import userdata

# Pass the Gemini key to the app through an environment variable, then start the server
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

<Popen: returncode: None args: ['streamlit', 'run', 'app.py', '--server.port...>

## 10.6 Get the public link

Run this cell after the app has started. It prints the link at which the app can be opened.

In [ ]:
import time
from pyngrok import ngrok

# Close old tunnels, then open a public link to the app (fixed address first, temporary link as fallback)
ngrok.kill()
time.sleep(3)

try:
    public_url = ngrok.connect(8501, domain="outsell-travel-reclusive.ngrok-free.dev")
except Exception:
    print("The fixed address is busy (an old session is still online). Using a temporary link instead.")
    public_url = ngrok.connect(8501)

print("Click this link:", public_url)

# Step 11: Summary and limitations

## Summary

- The system turns a plain-English question into SQL, validates it, runs it on a read-only database and explains the answer in plain English.
- On the 20-question test set it scored **20/20 (100%)**: 6/6 easy, 8/8 medium and 6/6 hard.
- Unsafe or unanswerable requests are stopped before anything is executed.
- The same logic is available as a Streamlit web app.

## Limitations

- The test covers one database (Northwind) and 20 questions, each with a single reference answer. A high score here does not guarantee the same accuracy on other databases.
- The scoring function does not compare row order, so a result with the right rows in the wrong order would still pass.
- The text safety check is deliberately strict. It rejects any statement that contains a keyword such as `REPLACE`, even where that word would be a harmless SQL function.
- Uploaded files in the web app were not part of the evaluation.
- The free Gemini plan limits the number of requests per minute, so the evaluation includes pauses between questions.

## Possible next steps

- Add more test questions, including ambiguous ones.
- Compare row order for questions where order matters.
- Test on a second database.